# A categorically organized CAS in Lean

This notebook is the acceptance proof of the CasDsl vertical slice: a
computer algebra system whose **user-facing interfaces are organized by the
mathematical categories where operations first make sense** — not by
implementation classes, and not by backends.

Everything below runs in a persistent Lean 4 kernel
([lean-jupyter-kernel](https://github.com/dzackgarza/lean-jupyter-kernel)).
Three invariants to watch for:

1. **Backend-blind syntax.** You will never see a backend named in an
   expression. Some results below are computed by SageMath through a direct
   typed adapter — the *developer's* routing configuration decides that,
   and `#explain_route` will show it. The mathematics doesn't change.
2. **Category-owned methods.** `factor`, `det`, `annihilator`, `nth` are
   declared on categories; objects receive them by membership and by
   *subcategory inheritance*, never by forwarding code on a leaf class.
3. **Semantic availability ≠ computability.** A method that makes
   mathematical sense stays available even when no implementation route
   exists yet — execution then fails with a *structured capability gap*
   (an auditable developer backlog item), never a fake value and never a
   type error. The final cell demonstrates this deliberately.


## 1 · Trusted arithmetic and assertions

`assert` is an *operational* assertion in the ordinary CAS sense: the
predicate is computed and trusted, with a fourfold outcome
`true | false | unknown | error`. Only `true` lets the cell commit.
No Lean theorem is generated, and no certificate is required — this is a
CAS, not a proof obligation machine.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/home/dzack/gitclones/lean-cas-dsl)…


1:0: ✓ 2 + 3 = 5


In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Backend-blind factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by **subcategory inheritance through the category
graph**, and is executed by whatever implementation the developer routed.


In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

The expression above never mentioned a backend. The routing that chose one
is developer diagnostics, not mathematics:


In [5]:
#explain_route n.factor()

1:0:   method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ


  method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ

## 3 · Polynomials, canonical maps, and calling a polynomial

`ℤ ⊆ ℚ` denotes the preferred canonical map — so `map p to ℚ[x]` moves a
polynomial along it without ceremony. And a polynomial can simply be
**called**: elaboration inserts evaluation through the preferred compatible
coefficient map. The mathematician writes `q(1)`, as on paper.

`map e to D` means: apply the preferred canonical map into `D` when one is
registered, and fail honestly otherwise. Canonical maps are *preferred
choices*, not necessarily injections — the inclusions `ℕ ⊆ ℤ ⊆ ℚ` are
monomorphisms, while `ℤ → ℤ/n` is the ring quotient, supplied by its
universal property. As of round two these are **registry data**: the
prelude registers them, the engine knows none of these facts, and an
unregistered pair fails with the honest `there is no preferred canonical
map` error — a missing coercion is never widened to a "reasonable"
conversion.

In [6]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


In [7]:
let q := map p to ℚ[x]

1:0: q := x^3 - 2x + 1 ∈ ℚ[x]


In [8]:
q.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [9]:
assert q(1) = 0

1:0: ✓ q(1) = 0


The quotient `ℤ → ℤ/n` is one registered rule for *every* modulus: an
integer names its residue class. `n` is still `360`, and `360 ≡ 3 (mod 7)`:

In [10]:
map n to ℤ/7

1:0: 3


3

## 4 · Exact matrix algebra

Matrix literals use row-semicolon syntax; `det` and `inverse` are methods
of the square-matrix category, computed exactly over `ℚ`.


In [11]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

1:0: M := [1, 2; 3, 4] ∈ Mat₂(ℚ)


In [12]:
M.inverse()

1:0: [-2, 1; 3/2, -1/2]


[-2, 1; 3/2, -1/2]

In [13]:
assert M.det() = -2

1:0: ✓ M.det() = -2


## 5 · Subcategory inheritance, for real

`annihilator : Modules(ℤ) → Ideals(ℤ)` is declared **once**, on the parent
category. `F` below is declared in the *proper subcategory*
`SmallModules(ℤ)` — which contains **no forwarding declaration**. The
method arrives purely through the registered inclusion
`SmallModules ≤ Modules`. (The ascription is doing real semantic work:
`ℤ/4` *in a module category* means the ℤ-module ℤ/4, not the ring.)


In [14]:
let F := ℤ/4 in SmallModules(ℤ)

1:0: F := ℤ/4 as ℤ-module


In [15]:
F.annihilator()

1:0: (4)


(4)

## 6 · Transport along preferred functors

`cardinality` is declared on `Sets` — and a module is not a set. But the
prelude registers the forgetful functor `UnderlyingSet : Modules(ℤ) → Sets`
as *preferred*, so the resolver transports the **receiver**:
`F.cardinality()` resolves as `UnderlyingSet(F).cardinality()`. Nothing was
declared on modules, no forwarding method exists anywhere, and the ordinary
call syntax is unchanged.

In [16]:
F.cardinality()

1:0: 4


4

Membership transports the same way — `2` names a residue class of the
underlying set:

In [17]:
assert 2 ∈ F

1:0: ✓ 2 ∈ F


Equality, by contrast, is **category-bound**. `U(F) = {0, 1, 2, 3}` in
Sets — but `F` itself is a module, and there is no *unique* module
structure on that set, so bare `=` between objects of different
categories is trivially false: it never inserts the functor. Comparing
them requires explicitly asking the question in a common comparison
category — which is exactly what the Sets method `set_eq` does (its
receiver transports, like `∈` above):

In [18]:
assert F ≠ {0, 1, 2, 3}

1:0: ✓ F ≠ {0, 1, 2, 3}


In [19]:
F.set_eq({0, 1, 2, 3})

1:0: true


true

Two guarantees, both machine-checked in the build:

- transport runs **only where direct resolution finds nothing** —
  `annihilator` above still arrives untransported through
  `SmallModules(ℤ) ≤ Modules(ℤ)`, so registering a functor can never take
  a method away from an object that already had it;
- two applicable functors would be an honest *ambiguity error* naming both,
  never a silent pick.

The transport step itself is developer diagnostics, not mathematics:

In [20]:
#explain_route F.cardinality()

1:0:   method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set


  method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set

## 7 · Countable sets, ellipses, and indexing

Countability is mathematical structure — a monomorphism into ℕ — not a
backend capability. A *registered enumeration choice* labels elements, so
countable objects support `nth` (`X[k]`, 0-based) and `cardinality`.
Ellipsis literals are the exact Haskell-style progressions, nothing more.

The registered convention for `ℤ` is `0, 1, −1, 2, −2, …` — a documented,
revisitable choice, never a claim that ℤ is intrinsically ordered that way.


In [21]:
let X := {0, 1, 2, ...}

1:0: X := {0, 1, ...}


In [22]:
assert X = ℕ

1:0: ✓ X = ℕ


In [23]:
let Y := {0, 2, 4, ...}

1:0: Y := {0, 2, ...}


In [24]:
assert 8 ∈ Y

1:0: ✓ 8 ∈ Y


In [25]:
assert 9 ∉ Y

1:0: ✓ 9 ∉ Y


In [26]:
ℤ[3]

1:0: 2


2

In [27]:
X.cardinality()

1:0: ℵ₀


ℵ₀

## 8 · Semantic availability is not computability

`ℚ` is countable, so `nth` is *semantically* available on it — the category
layer says so, and no implementation hole is allowed to redefine the
mathematics (there is deliberately no `EnumerableCountableSet` category
here). But the developer has not yet registered an enumeration route for
`ℚ`. The audit surface shows the hole as structured backlog:


In [28]:
#capability_gaps

1:0:   representative    method        category              status
  ℚ                 nth           CountableSets(ℚ)      no route matches this presentation (6 registered for the method, none matching)
                                  available by: declared directly on CountableSets(ℚ)
  x^3 − 2x + 1 ∈ ℤ[x] factor        FactorizationElems(ℤ[x]) no route matches this presentation (2 registered for the method, none matching)
                                  available by: declared directly on FactorizationElems(ℤ[x])
  24 method/representative pair(s) are implemented; 2 are backlog.


  representative    method        category              status
  ℚ                 nth           CountableSets(ℚ)      no route matches this presentation (6 registered for the method, none matching)
                                  available by: declared directly on CountableSets(ℚ)
  x^3 − 2x + 1 ∈ ℤ[x] factor        FactorizationElems(ℤ[x]) no route matches this presentation (2 registered for the method, none matching)
                                  available by: declared directly on FactorizationElems(ℤ[x])
  24 method/representative pair(s) are implemented; 2 are backlog.

So the next cell **fails on purpose** — with a structured
`NoImplementation` capability gap naming the method, the receiver
category, the presentation, and the routes considered. Not a parse error,
not a type error, and not a silent lie. This failing cell is part of the
proof.


In [29]:
ℚ[3]

LeanError: NoImplementation: 'nth' is mathematically available here, but no registered route can execute it for this presentation.
  method:            nth
  receiver category: CountableSets(ℚ)
  presentation:      ℚ
  semantic path:     declared directly on CountableSets(ℚ)
  routes considered: 6
    - nth for the domain ℕ → backend native, op "nth", priority 0
    - nth for the underlying set of ℕ → backend native, op "nth", priority 0
    - nth for the domain ℤ → backend native, op "nth", priority 0
    - nth for the underlying set of ℤ → backend native, op "nth", priority 0
    - nth for a progression over _ → backend native, op "nth", priority 0
    - nth for a finite set → backend native, op "nth", priority 0
This is a developer backlog item, not a narrowing of the mathematics: the method stays available on the category.